# 02.01 Data Quality and Verification

**Phase 02 — Data Preparation**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Assess data completeness, schema consistency, duplicate records, distribution anomalies, and verify data integrity before any transformation.


### Research Objectives

1. Identify missing values, outliers, invalid entry formats, and contradictory records.
2. Check for duplicate patient registrations and conflicting assessment entries.
3. Verify data types, range constraints (e.g., age >= 0, monthly income >= 0), and code validities.
4. Generate a data quality score matrix for all candidate predictor variables.


### Inputs & Expected Outputs

**Required Inputs:**
- `Raw patient tabular datasets (`data/raw/tabular/`).`
- `Inventory manifests from Phase 01 (`data/interim/validated/source_inventory_manifest.json`).`

**Expected Outputs:**
- `Data quality report summary (`data/interim/validated/data_quality_report.json`).`
- `Identification log of records requiring sanitization or imputation.`


### 1. Environment Setup & Library Imports

In [ ]:
from pathlib import Path
import json, datetime, re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())

### 1. Setup Environment and Load Input Manifests

In [ ]:
SYNTH_CSV = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'data_quality' / 'missingness_profile.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(SYNTH_CSV)
print(f'Loaded {len(df)} records x {df.shape[1]} columns from {SYNTH_CSV.name}')

### 2. Completeness and Missing Value Audit

In [ ]:
profile_rows = []
for col in df.columns:
    null_cnt = int(df[col].isna().sum())
    profile_rows.append({
        'column_name': col,
        'dtype': str(df[col].dtype),
        'null_count': null_cnt,
        'null_percentage': round((null_cnt / len(df)) * 100.0, 2),
        'non_null_count': int(len(df) - null_cnt),
        'source_dataset': SYNTH_CSV.name,
    })
profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv(AUDIT_CSV_PATH, index=False)
print(f'Saved missingness profile ({len(profile_df)} columns) -> {AUDIT_CSV_PATH}')

### 3. Duplicate and Inconsistency Detection

In [ ]:
dup_count = int(df.duplicated().sum())
print(f'Exact duplicate rows: {dup_count}')
print('Target distribution:', df['eligibility'].value_counts(dropna=False).to_dict())

### 4. Outlier and Value Range Verification

In [ ]:
age_bad = int(((df['age'] < 0) | (df['age'] > 120)).sum())
hs_bad = int((pd.to_numeric(df['household_size'], errors='coerce').dropna() < 1).sum())
print(f'Range check -> invalid ages: {age_bad}, invalid household_size (<1): {hs_bad}')

### Findings & Observations

- **Observations:** Summarize initial data quality metrics: missingness percentages, duplicate rates, and anomalous records.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Missing value distributions quantified per attribute.
- [ ] Duplicate records audited with deterministic criteria.
- [ ] Range bounds validated for all numerical clinical/socioeconomic variables.
- [ ] Data quality summary generated.
